In [6]:
##### LOCAL or INTERACTIVE NODES #####
import deeplabcut

deeplabcut.create_new_project(
    "Cheeseboard",
    "Deeplabcut_analyze",
    ["Z:/forgetting/Carla/Cheeseboard/APPPS1/AHAD01.37/2months/Post/28_02_2025/AHAD01.37-AfterTest1.mp4",],
    working_directory="//10.69.168.1/crnldata/forgetting/Carla/",
    copy_videos=False,
    multianimal=False
)

Created "\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\videos"
Created "\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\labeled-data"
Created "\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\training-datasets"
Created "\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.
Z:\forgetting\Carla\Cheeseboard\APPPS1\AHAD01.37\2months\Post\28_02_2025\AHAD01.37-AfterTest1.mp4 copied to \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\videos\AHAD01.37-AfterTest1.mp4
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\videos\AHAD01.37-AfterTest1.mp4
Generated "\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml"

A new pr

'\\\\10.69.168.1\\crnldata\\forgetting\\Carla\\Cheeseboard-Deeplabcut_analyze-2026-07-31\\config.yaml'

In [8]:
import os
import re
import random
import yaml
from collections import defaultdict

config = r'Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml'
basepath = r'Z:\forgetting\Carla\Cheeseboard'

SESSION_TYPES = ['Habituation', 'Training', 'Post', 'Test']  # ordre = priorité de détection
ANIMAL_PATTERN = re.compile(r'AHAD\d{2}\.\d{2}', re.IGNORECASE)

def detect_animal(path):
    m = ANIMAL_PATTERN.search(path)
    return m.group(0).upper() if m else None

def detect_session(path):
    parts = path.replace('\\', '/').split('/')
    # priorité : nom des dossiers (structure Animal/age/SessionType/date/fichier.mp4)
    for part in parts[:-1]:
        low = part.lower()
        for cat in SESSION_TYPES:
            if cat.lower() in low:
                return cat
    # fallback : nom du fichier (anciennes vidéos "Training J1-3.mp4" sans dossier dédié)
    filename = parts[-1].lower()
    for cat in SESSION_TYPES:
        if cat.lower() in filename:
            return cat
    return None

# 1. Vidéos déjà dans le projet
with open(config, 'r') as f:
    cfg = yaml.safe_load(f)
existing_videos = set(os.path.basename(p) for p in cfg['video_sets'].keys())

# 2. Scan de toutes les vidéos disponibles + classification
groups = defaultdict(list)   # (animal, session_type) -> [chemins]
unclassified = []

for root, dirs, files in os.walk(basepath):
    for file in files:
        if not file.lower().endswith(".mp4") or "Cheeseboard" not in root:
            continue
        full_path = os.path.join(root, file)
        if file in existing_videos:
            continue  # déjà dans le projet, on ignore

        animal = detect_animal(full_path)
        session = detect_session(full_path)

        if animal and session:
            groups[(animal, session)].append(full_path)
        else:
            unclassified.append(full_path)

# 3. Tirage aléatoire : 1 vidéo par (animal, session_type)
random.seed(42)  # retirez cette ligne si vous voulez un tirage différent à chaque exécution
selected = []
for (animal, session), paths in sorted(groups.items()):
    pick = random.choice(paths)
    selected.append(pick)

# 4. Rapport
animals = sorted(set(a for a, s in groups.keys()))
sessions_found = sorted(set(s for a, s in groups.keys()))

print(f"📊 Animaux détectés        : {len(animals)} → {animals}")
print(f"📊 Types de session trouvés: {sessions_found}")
print(f"🎲 Vidéos sélectionnées    : {len(selected)}")
print(f"⚠️  Fichiers non classés    : {len(unclassified)}")

print("\n🔍 Détail de la sélection :")
for (animal, session), paths in sorted(groups.items()):
    print(f"   {animal:12s} | {session:12s} | {len(paths)} candidate(s)")

print("\n✅ Vidéos retenues :")
for v in selected:
    print(f"   {v}")

if unclassified:
    print("\n⚠️  Non classées (à vérifier manuellement) :")
    for v in unclassified[:10]:
        print(f"   {v}")
    if len(unclassified) > 10:
        print(f"   ... et {len(unclassified) - 10} autres")

📊 Animaux détectés        : 37 → ['AHAD01.37', 'AHAD01.38', 'AHAD01.39', 'AHAD01.40', 'AHAD01.41', 'AHAD01.42', 'AHAD01.43', 'AHAD01.44', 'AHAD02.01', 'AHAD02.02', 'AHAD02.03', 'AHAD02.04', 'AHAD02.05', 'AHAD02.06', 'AHAD02.07', 'AHAD02.08', 'AHAD11.10', 'AHAD11.58', 'AHAD11.59', 'AHAD11.60', 'AHAD11.61', 'AHAD11.62', 'AHAD11.63', 'AHAD11.64', 'AHAD11.65', 'AHAD11.66', 'AHAD11.67', 'AHAD11.68', 'AHAD11.69', 'AHAD11.70', 'AHAD11.71', 'AHAD11.72', 'AHAD11.73', 'AHAD11.74', 'AHAD11.75', 'AHAD12.10', 'AHAD21.15']
📊 Types de session trouvés: ['Habituation', 'Post', 'Test', 'Training']
🎲 Vidéos sélectionnées    : 139
⚠️  Fichiers non classés    : 0

🔍 Détail de la sélection :
   AHAD01.37    | Post         | 32 candidate(s)
   AHAD01.37    | Test         | 11 candidate(s)
   AHAD01.37    | Training     | 44 candidate(s)
   AHAD01.38    | Post         | 33 candidate(s)
   AHAD01.38    | Test         | 11 candidate(s)
   AHAD01.38    | Training     | 45 candidate(s)
   AHAD01.39    | Habituati

In [9]:
deeplabcut.add_new_videos(config, selected, copy_videos=True)

Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the 

In [ ]:
import deeplabcut
deeplabcut.add_new_videos(
    "C:/Users/AudreyHay/Documents/Carla/Cheeseboard-Deeplabcut_analyze-2026-07-31/config.yaml",
  [    
        
    ],
    copy_videos=True
)


Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
Copying the videos
New videos were added to the project! Use the function 'extract_frames' to select frames for labeling.


In [16]:
deeplabcut.extract_frames(
    r"Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml",
    mode="automatic",
    algo="uniform",
    crop=False,
    userfeedback=False
)


Config file read successfully.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 38.33  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 52.77  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 189.45  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 25.69  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 89.77  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 190.96  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 35.58  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 472.05  seconds.
Extracting frames based on uniform ...
Uniformly extracting of frames from 0.0  seconds to 49.73  seconds.
Ext

In [16]:
import deeplabcut
deeplabcut.label_frames(r"Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml")

In [ ]:
deeplabcut.check_labels(r"Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml", visualizeindividuals=False)

In [ ]:
import deeplabcut

path_config_file = r"Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml"

deeplabcut.create_training_dataset(path_config_file, Shuffles=[1], net_type="resnet_50", augmenter_type="imgaug")
deeplabcut.train_network(path_config_file, shuffle=1, saveiters=5000, displayiters=100, maxiters=300000, allow_growth=True)
deeplabcut.evaluate_network(path_config_file, Shuffles=[1], plotting=True)

\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\labeled-data\AHAD01.41-Training J2-5\CollectedData_Deeplabcut_analyze.h5  not found (perhaps not annotated).
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\labeled-data\AHAD01.42-AfterTest9-2\CollectedData_Deeplabcut_analyze.h5  not found (perhaps not annotated).
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\labeled-data\AHAD01.42-Test6\CollectedData_Deeplabcut_analyze.h5  not found (perhaps not annotated).
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\labeled-data\AHAD01.42-Training J2-5\CollectedData_Deeplabcut_analyze.h5  not found (perhaps not annotated).
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\labeled-data\AHAD01.43-AfterTest6-3\CollectedData_Deeplabcut_analyze.h5  not found (perhaps not annotated).
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Dee

Training with configuration:
data:
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [1.0, 1.0]
      translation: 0
    collate:
      type: ResizeFromDataSizeCollate
      min_scale: 0.4
      max_scale: 1.0
      min_short_side: 128
      max_short_side: 1152
      multiple_of: 32
      to_square: False
    covering: False
    gaussian_noise: 12.75
    hist_eq: False
    motion_blur: False
    normalize_images: True
device: auto
metadata:
  project_path: \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31
  pose_config_path: \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\dlc-models-pytorch\iteration-0\CheeseboardJul31-trainset95shuffle1\train\pytorch_config.yaml
  bodyparts: ['bodypart1', 'bodypart2', 'bodypart3', 'objectA']
  unique_bodyparts: []
  individuals: ['animal']
  with_identity: None
method: bu
model:
  backbone:
    type: ResNet
    

In [2]:
import deeplabcut
path_config_file = r"Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml"
deeplabcut.evaluate_network(path_config_file, Shuffles=[1], plotting=True)

Loading DLC 3.0.0rc6...


100%|██████████| 28/28 [00:34<00:00,  1.22s/it]


Evaluation results for DLC_Resnet50_CheeseboardJul31shuffle1_snapshot_190-results.csv (pcutoff: 0.6):
train rmse             0.67
train rmse_pcutoff     0.67
train mAP             96.92
train mAR             98.41
test rmse              1.29
test rmse_pcutoff      1.26
test mAP              78.57
test mAR              85.56
Name: (0.95, 1, 190, -1, 0.6), dtype: float64


In [4]:
import deeplabcut
import pandas as pd
import numpy as np

config_path = r"Z:\forgetting\Carla\Cheeseboard-Deeplabcut_analyze-2026-07-31\config.yaml"

# plotting=True sauvegarde aussi des scatter plots prédits vs labellisés PAR bodypart
# dans evaluation-results/iteration-X/...-plots/
deeplabcut.evaluate_network(config_path, plotting=True, per_keypoint_evaluation=True)

100%|██████████| 28/28 [02:03<00:00,  4.43s/it]


Evaluation results for DLC_Resnet50_CheeseboardJul31shuffle1_snapshot_190-results.csv (pcutoff: 0.6):
train rmse             0.67
train rmse_pcutoff     0.67
train mAP             96.92
train mAR             98.41
test rmse              1.29
test rmse_pcutoff      1.26
test mAP              78.57
test mAR              85.56
Name: (0.95, 1, 190, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_CheeseboardJul31shuffle1_snapshot_190-keypoint-results):
  Train error (px)
    bodypart1:   0.72px
    bodypart2:   0.65px
    bodypart3:   0.64px
    objectA:     nanpx
  Test error (px)
    bodypart1:   1.60px
    bodypart2:   0.98px
    bodypart3:   1.22px
    objectA:     nanpx


In [7]:
gt = pd.read_hdf("Z:/forgetting/Carla/Cheeseboard-Deeplabcut_analyze-2026-07-31/labeled-data/AHAD01.37-AfterTest1/CollectedData_Deeplabcut_analyze.h5")
pred = pd.read_hdf("Z:/forgetting/Carla/Cheeseboard-Deeplabcut_analyze-2026-07-31/evaluation-results-pytorch/iteration-0/CheeseboardJul31-trainset95shuffle1/DLC_Resnet50_CheeseboardJul31shuffle1_snapshot_190.h5")  # prédictions machine

# Aligne pred sur les frames labellisées uniquement
pred = pred.reindex(gt.index)

# Vérification rapide que ça matche
print(len(gt), len(pred), pred.isna().all(axis=1).sum())  # le dernier doit être 0

bodyparts = gt.columns.get_level_values("bodyparts").unique()
for bp in bodyparts:
    gx = gt.xs((bp, "x"), level=("bodyparts","coords"), axis=1)
    gy = gt.xs((bp, "y"), level=("bodyparts","coords"), axis=1)
    px = pred.xs((bp, "x"), level=("bodyparts","coords"), axis=1)
    py = pred.xs((bp, "y"), level=("bodyparts","coords"), axis=1)
    err = np.sqrt((gx.values - px.values)**2 + (gy.values - py.values)**2)
    print(f"{bp}: RMSE = {np.nanmean(err):.2f} px  (n={np.sum(~np.isnan(err))})")

20 20 0
bodypart1: RMSE = 0.65 px  (n=16)
bodypart2: RMSE = 0.60 px  (n=14)
bodypart3: RMSE = 0.63 px  (n=16)
objectA: RMSE = nan px  (n=0)


C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_27212\118045191.py:17: RuntimeWarning: Mean of empty slice
  print(f"{bp}: RMSE = {np.nanmean(err):.2f} px  (n={np.sum(~np.isnan(err))})")


In [8]:
gx_valid = gt.xs((bp, "x"), level=("bodyparts","coords"), axis=1).notna().sum()
px_valid = pred.xs((bp, "x"), level=("bodyparts","coords"), axis=1).notna().sum()
print(f"objectA — labels manuels valides: {gx_valid.values}, prédictions valides: {px_valid.values}")

objectA — labels manuels valides: [0], prédictions valides: [20]


In [12]:
from pathlib import Path
import numpy as np
import deeplabcut
from deeplabcut.core.weight_init import WeightInitialization
from deeplabcut.utils import auxiliaryfunctions

config_path = "chemin/vers/config.yaml"
cfg = auxiliaryfunctions.read_config(config_path)
bodyparts = cfg["bodyparts"]

# Mapping identité : chaque bodypart correspond à lui-même (même ordre, même projet)
conversion_array = np.arange(len(bodyparts))

weight_init = WeightInitialization(
    snapshot_path=Path("chemin/vers/dlc-models-pytorch/iteration-0/.../train/snapshot-190.pt"),
    with_decoder=True,
    conversion_array=conversion_array,
    bodyparts=bodyparts,
)

deeplabcut.create_training_dataset(
    config_path,
    net_type="resnet_50",
    weight_init=weight_init,
)

FileNotFoundError: Config file at chemin\vers\config.yaml not found. Please make sure that the file exists and/or that you passed the path of the config file correctly!